# 1. Create HiSeq Object and Initialize Instruments. 


If cameras take longer than 2 min. to initialize
1. Power off the HiSeq
2. Restart the Computer
3. Turn on the HiSeq and try again

In [1]:
import pyseq
from pyseq import focus
import threading
import os
from datetime import datetime
from pathlib import Path

hs = pyseq.HiSeq() # Create HiSeq Object
hs.initializeCams() # Initialize Cameras
hs.initializeInstruments() # Initialize all other instruments (stages, pumps, valve, etc.)

2026-05-18 14:55 - HiSeq::Initializing cameras 
2026-05-18 14:55 - HiSeq::Initializing FPGA 
2026-05-18 14:55 - HiSeq::Initializing X & Y stages 
2026-05-18 14:56 - HiSeq::Initializing lasers 
2026-05-18 14:56 - HiSeq::Initializing pumps and valves 
2026-05-18 14:57 - HiSeq::Initializing optics and Z stages 
2026-05-18 14:58 - HiSeq::Syncing Y stage 
2026-05-18 14:58 - HiSeq::Initialized!


True

# 2.  Configure Instrument

Configure image directory and the ROI (region of interest to image)

In [2]:
flowcell = "A" # flow cell slot, can be "A" or "B" 
output_dir = os.path.expanduser('~') # Default is the user home directory / tetraspeck_YYYYMMDD
roi = [13, 35.5, 12, 34.5] # measure: Lower Left x, Lower Left y, Upper Right x, Upper Right y

# ROI to image
section = "tetraspeck"
pos_dict = hs.position(flowcell, roi)

# Configure HiSeq

## Make image and focus paths
f_date = datetime.now().strftime("%Y%m%d")
hs.image_path = Path(output_dir) / f"{section}_{f_date}"
hs.focus_path = Path(output_dir) / f"{section}_{f_date}/focus"
os.makedirs(hs.focus_path, exist_ok=True)

## Use full once autofocus routine
hs.AF = "full once" # options are full once, once, partial once, and partial

## Write machine name to image and focus path
model, name, focus_path = pyseq.methods.get_machine_info()
hs.name = name
for p in [hs.image_path, hs.focus_path]:
    with open(p/"machine_name.txt", "w") as file:
        file.write(hs.name)


# Move stage Out
hs.move_stage_out() 


2026-05-18 14:58 - HiSeq::Moving stage out 


# 3. Autofocus

**Make sure to place tetraspeck slide on `flowcell` slot on stage.**

Finds optimal objective focal position 

Uses 100 mW laser power to focus, which is relatively high because tetraspeck beads are dim

Autofocus TIFF images and data will be written to `hs.focus_path`. Images will be deleted after focus routine completes. 

## Common Warnings
- Spacing too small, changing to  4.020266666666666
- OBJstage::WARNING:: Could not read objective position

In [3]:
# Set optical parameters for focusing
print("Setting Optical Parameters")
for color in ["red", "green"]:
    hs.lasers[color].set_power(100)
    hs.optics.move_ex(color, 'open')

# Move stages to initial imaging position
print("Moving Stages")
hs.y.move(pos_dict['y_initial'])
hs.x.move(pos_dict['x_initial'])
hs.z.move([21800, 21800, 21800])
hs.obj.move(hs.obj.focus_rough)

# Run autofocus routine
opt_obj_pos = focus.autofocus(hs, pos_dict)
hs.obj.move(opt_obj_pos) # move objective to optimal position

Setting Optical Parameters
Moving Stages


2026-05-18 15:00 - Autofocus::FullScan::Scanning section 
2026-05-18 15:00 - HiSeq::Scan::Tile 1 / 2 
2026-05-18 15:01 - HiSeq::Scan::Tile 2 / 2 
2026-05-18 15:01 - Autofocus::FullScan::Stitching & Normalizing images 
2026-05-18 15:01 - Autofocus::Analyzing out of focus image 
2026-05-18 15:01 - Autofocus::Finding potential focus positions 
2026-05-18 15:01 - Autofocus::Found 42 focus positions 
2026-05-18 15:01 - Autofocus::Finding optimal focus 


Spacing too small, changing to  4.020266666666666


2026-05-18 15:02 - OBJstage::WARNING:: Could not read objective position
2026-05-18 15:03 - GetFocusData::Found focus point 1
2026-05-18 15:03 - OBJstage::WARNING:: Could not read objective position
2026-05-18 15:03 - GetFocusData::Found focus point 2
2026-05-18 15:04 - OBJstage::WARNING:: Could not read objective position
2026-05-18 15:04 - GetFocusData::Found focus point 3
2026-05-18 15:04 - OBJstage::WARNING:: Could not read objective position
2026-05-18 15:04 - GetFocusData::Found focus point 4
2026-05-18 15:05 - OBJstage::WARNING:: Could not read objective position
2026-05-18 15:05 - GetFocusData::Found focus point 5
2026-05-18 15:05 - Autofocus::Completed in 4 minutes 


# 4. Image ROI

Images `nz` of z planes in roi

TIFF images will be written to `hs.image_path`

outputs time (s) HiSeq took to scan tetraspeck ROI

In [4]:
# Image parameters
nz = 20 # Number of z planes to image

# Optical parameters
laser_power = 200 #mW
em_filter = "open" 

print("Setting Optical Parameters")
for color in ["red", "green"]:
    hs.lasers[color].set_power(laser_power)
    hs.optics.move_ex(color, em_filter)
    
print("Moving Stages")
hs.y.move(pos_dict['y_initial'])
hs.x.move(pos_dict['x_initial'])
z_initial = opt_obj_pos - (nz // 2 * hs.nyquist_obj)
hs.obj.move(z_initial)

print(f'Imaging {section}')   
im_name = f'{flowcell}_s{section}_r1' # flowcell_section_round
hs.image_pth = Path(output_dir) # Reset image path after autofocus
hs.scan(nz, pos_dict=pos_dict, image_name=im_name)

Setting Optical Parameters
Moving Stages


2026-05-18 15:06 - HiSeq::Scan::Tile 1 / 2 


Imaging tetraspeck


2026-05-18 15:15 - HiSeq::Scan::Tile 2 / 2 


1080.4976496696472

# 5. Reset HiSeq

In [5]:
print("Reseting X-Stage")
hs.x.move(30000) # center x stage
print("Reseting Y-Stage")
hs.y.move(0) # center y stage,
hs.y.command("OFF") # turn off y stage
print("Reseting Z-Stage")
hs.z.move([0, 0, 0]) # lower tilt stage
print("Reseting Lasers and Filters")
for color in ["red", "green"]:
    hs.optics.move_ex(color, 'home') # home emission filters
    hs.lasers[color].turn_on(False) # Turn off lasers
print("Reset complete")

Reseting X-Stage
Reseting Y-Stage
Reseting Z-Stage
Reseting Lasers and Filters
Reset complete


## If using HiSeq within 10 days after reset, leave the system on.
1. Lock flow cells on to the stage and submerge all line inlets in water to prevent them from drying out.

2. Close the stage door.

3. Idle the sequencer: ```pyseq -idle```

## If not using HiSeq in more than 10 day after reset, turn off the sytem.
1. Remove the flow cells from the stage and all reagents from the chiller

2. Close the stage door

3. Turn the power off to the HiSeq